In [ ]:
import numpy as np
import pandas as pd
import os

def filter_admet_candidates(
    df: pd.DataFrame,
    *,
    cns_target: bool = True,      
    aff_hard: float = -8.0,
    mw_max_cns: float = 480.0,
    mw_max_non_cns: float = 500.0,
    sa_cut_screen: float = 5.0,
    sa_cut_candidate: float = 4.5,
    stage: str = "screen",        # "screen" 或 "candidate"
) -> pd.DataFrame:
    """
    通用 ADMET 筛选函数：支持 CNS / 非 CNS 项目。

    需要 df 至少包含（如存在）：
    QED, logP, MW, SA, aff,
    hERG, DILI, AMES, CYP3A4_Veith, CYP2D6_Veith,
    HIA_Hou, Caco2_Wang, BBB_Martins

    cns_target:
        True  -> CNS 项目（希望 BBB 穿透高）；
        False -> 非 CNS / 外周项目（希望 BBB 穿透低）。
    stage:
        "screen"    -> 初筛（稍宽）
        "candidate" -> 候选集（更严格的 SA / 毒性）
    """
    df = df.copy()

    # ---------- 1. 物化 + docking 硬过滤 ----------
    # QED: 通用阈值
    if "QED" in df:
        df["pass_QED"] = df["QED"] >= 0.6
    else:
        df["pass_QED"] = True

    # logP / MW: CNS vs 非 CNS 不同窗口
    if cns_target:
        logp_low, logp_high = 1.5, 4.0
        mw_max = mw_max_cns
    else:
        logp_low, logp_high = 0.0, 5.0
        mw_max = mw_max_non_cns

    if "logP" in df:
        df["pass_logP"] = (df["logP"] >= logp_low) & (df["logP"] <= logp_high)
    else:
        df["pass_logP"] = True

    if "MW" in df:
        df["pass_MW"] = df["MW"] <= mw_max
    else:
        df["pass_MW"] = True

    sa_cut = sa_cut_candidate if stage == "candidate" else sa_cut_screen
    if "SA" in df:
        df["pass_SA"] = df["SA"] <= sa_cut
    else:
        df["pass_SA"] = True

    if "aff" in df:
        df["pass_aff"] = df["aff"] <= aff_hard
    else:
        df["pass_aff"] = True

    # ---------- 2. 毒性端点：hard vs preferred ----------
    def tox_hard(p):
     
        return p <= 0.7

    def tox_pref(p, tight=False):
      
        if tight:
            return p <= (0.3 if stage == "candidate" else 0.4)
     
        return p <= (0.5 if stage == "candidate" else 0.6)

    for col in ["hERG", "DILI"]:
        if col in df:
            df[f"{col}_hard"] = df[col].apply(tox_hard)
            df[f"{col}_pref"] = df[col].apply(lambda p: tox_pref(p, tight=True))
        else:
            df[f"{col}_hard"] = True
            df[f"{col}_pref"] = True

    if "AMES" in df:
        df["AMES_hard"] = df["AMES"].apply(tox_hard)
        df["AMES_pref"] = df["AMES"].apply(tox_pref)
    else:
        df["AMES_hard"] = True
        df["AMES_pref"] = True

    if {"CYP3A4_Veith", "CYP2D6_Veith"}.issubset(df.columns):
        df["CYP_max"] = df[["CYP3A4_Veith", "CYP2D6_Veith"]].max(axis=1)
        df["CYP_hard"] = df["CYP_max"] <= 0.7
        df["CYP_pref"] = df["CYP_max"] <= 0.5
    else:
        df["CYP_max"] = np.nan
        df["CYP_hard"] = True
        df["CYP_pref"] = True

  
    def adme_hard_abs(p):
        return p >= 0.5

    def adme_pref_abs(p):
        return p >= 0.7

    if "HIA_Hou" in df:
        df["HIA_hard"] = df["HIA_Hou"].apply(adme_hard_abs)
        df["HIA_pref"] = df["HIA_Hou"].apply(adme_pref_abs)
    else:
        df["HIA_hard"] = True
        df["HIA_pref"] = True

    if "Caco2_Wang" in df:
        df["Caco2_hard"] = df["Caco2_Wang"].apply(adme_hard_abs)
        df["Caco2_pref"] = df["Caco2_Wang"].apply(adme_pref_abs)
    else:
        df["Caco2_hard"] = True
        df["Caco2_pref"] = True

    if "BBB_Martins" in df:

        if cns_target:
            def bbb_hard(p):
                return p >= 0.6    
            def bbb_pref(p):
                return p >= 0.7     
        else:
            def bbb_hard(p):
                return p <= 0.6     
            def bbb_pref(p):
                return p <= 0.5     

        df["BBB_hard"] = df["BBB_Martins"].apply(bbb_hard)
        df["BBB_pref"] = df["BBB_Martins"].apply(bbb_pref)

    else:
        df["BBB_hard"] = True
        df["BBB_pref"] = True

    hard_cols = [
        "pass_QED", "pass_logP", "pass_MW", "pass_SA", "pass_aff",
        "hERG_hard", "DILI_hard", "AMES_hard",
        "CYP_hard", "HIA_hard", "Caco2_hard", "BBB_hard",
    ]
    hard_cols = [c for c in hard_cols if c in df.columns]
    df["keep_hard"] = df[hard_cols].all(axis=1)

    pref_cols = [
        "hERG_pref", "DILI_pref", "AMES_pref",
        "CYP_pref", "HIA_pref", "Caco2_pref", "BBB_pref",
    ]
    pref_cols = [c for c in pref_cols if c in df.columns]
    df["keep_strict"] = df["keep_hard"] & df[pref_cols].all(axis=1)

    return df


def run_filter_and_save(
    input_csv: str,
    *,
    vina_score : float = -8.0,
    cns_target: bool = True,
    output_all: str = None,
    output_keep_hard: str = None,
    output_keep_strict: str = None,
    output_incomplete: str = None,  # 新增：保存缺失关键列的行
    stage: str = "screen",
    **filter_kwargs,
):
   
    df = pd.read_csv(input_csv)

    critical_cols = [
        "aff",
        "hERG", "DILI", "AMES",
        "CYP3A4_Veith", "CYP2D6_Veith",
        "HIA_Hou", "Caco2_Wang", "BBB_Martins"
    ]

    existing_critical = [col for col in critical_cols if col in df.columns]

    if not existing_critical:
        df_complete = pd.DataFrame()
        df_incomplete = df.copy()
    else:
        mask_complete = df[existing_critical].notna().all(axis=1)
        df_complete = df[mask_complete].copy()
        df_incomplete = df[~mask_complete].copy()

    if output_incomplete is not None:
        df_incomplete.to_csv(output_incomplete, index=False)

    if len(df_complete) > 0:
        df_flagged = filter_admet_candidates(
            df_complete,
            cns_target=cns_target,
            aff_hard=vina_score,
            stage=stage,
            **filter_kwargs,
        )

        df_hard = df_flagged[df_flagged["keep_hard"]].copy()
        df_strict = df_flagged[df_flagged["keep_strict"]].copy()

        # 保存结果
        if output_all is not None:
            df_flagged.to_csv(output_all, index=False)
            print(f"✅ 已保存带标签的完整数据 ({len(df_flagged)} 行) 到: {output_all}")

        if output_keep_hard is not None:
            df_hard.to_csv(output_keep_hard, index=False)
            print(f"✅ 已保存 keep_hard ({len(df_hard)} 行) 到: {output_keep_hard}")

        if output_keep_strict is not None:
            df_strict.to_csv(output_keep_strict, index=False)
            print(f"✅ 已保存 keep_strict ({len(df_strict)} 行) 到: {output_keep_strict}")

        return df_flagged, df_hard, df_strict, df_incomplete
    else:
        print("ℹ️  无完整行可用于过滤")
        empty_flagged = pd.DataFrame()
        return empty_flagged, empty_flagged, empty_flagged, df_incomplete


In [22]:
input_file = "/raid/home/xukai/FRATTVAE/results/CNS_SMILES_standardized_struct_1020/generate/generate_best_1_hHPK1_properties_updated_admet.csv"
current_dir = os.getcwd()
basename = os.path.splitext(os.path.basename(input_file))[0]
# output_file = os.path.join(current_dir, basename+"_flaged_filtered_molecules_lrrk2.csv") 
output_keep_hard = os.path.join(current_dir, "sos1_hard_"+basename+"_flaged_filtered_molecules.csv") 
output_keep_strict = os.path.join(current_dir, "sos1_strict_"+basename+"_flaged_filtered_molecules.csv") 

df_all, df_hard, df_strict, df_incomplete = run_filter_and_save(
    input_file,
    vina_score=-8.0,
    cns_target=False,                       # CNS 项目
    stage="screen",
    # output_all=output_file,
    output_keep_hard=output_keep_hard,
    output_keep_strict=output_keep_strict,
    output_incomplete=f"sos1_incomplete_{basename}_records.csv",
)
df_all, df_hard, df_strict, df_incomplete = run_filter_and_save(
    input_file,
    vina_score=-8.0,
    cns_target=False,                       # CNS 项目
    stage="candidate",
    # output_all=output_file,
    output_keep_hard=output_keep_hard,
    output_keep_strict=output_keep_strict,
    output_incomplete=f"sos1_incomplete_{basename}_records.csv",
)
print(sum(df_hard['BBB_pref'] == 0) / len(df_hard) * 100)
print(sum(df_hard['Caco2_pref'] == 0) / len(df_hard) * 100)

🔍 检查的关键列（共 9 个）: ['aff', 'hERG', 'DILI', 'AMES', 'CYP3A4_Veith', 'CYP2D6_Veith', 'HIA_Hou', 'Caco2_Wang', 'BBB_Martins']
📊 总行数: 10018 | 完整行: 10018 | 缺失关键列: 0
💾 已保存 0 条 incomplete 记录到: sos1_incomplete_generate_best_1_hHPK1_properties_updated_admet_records.csv
✅ 已保存 keep_hard (48 行) 到: /raid/home/xukai/FRATTVAE_Analysis/sos1_hard_generate_best_1_hHPK1_properties_updated_admet_flaged_filtered_molecules.csv
✅ 已保存 keep_strict (0 行) 到: /raid/home/xukai/FRATTVAE_Analysis/sos1_strict_generate_best_1_hHPK1_properties_updated_admet_flaged_filtered_molecules.csv
🔍 检查的关键列（共 9 个）: ['aff', 'hERG', 'DILI', 'AMES', 'CYP3A4_Veith', 'CYP2D6_Veith', 'HIA_Hou', 'Caco2_Wang', 'BBB_Martins']
📊 总行数: 10018 | 完整行: 10018 | 缺失关键列: 0
💾 已保存 0 条 incomplete 记录到: sos1_incomplete_generate_best_1_hHPK1_properties_updated_admet_records.csv
✅ 已保存 keep_hard (48 行) 到: /raid/home/xukai/FRATTVAE_Analysis/sos1_hard_generate_best_1_hHPK1_properties_updated_admet_flaged_filtered_molecules.csv
✅ 已保存 keep_strict (0 行) 到: /raid/ho

In [18]:
input_file = "/raid/home/xukai/FRATTVAE/results/chembl_36_20251023_r1r10w1100w900_standardized_f_dyn_oral_struct_1031/generate/generate_oral_pg_hpk1_admet_39_dyn_oral_policy_200_20251107_115825_0_hHPK1_vina_admet.csv"
current_dir = os.getcwd()
basename = os.path.splitext(os.path.basename(input_file))[0]
# output_file = os.path.join(current_dir, basename+"_flaged_filtered_molecules_lrrk2.csv") 
output_keep_hard = os.path.join(current_dir, "sos1_hard_"+basename+"_flaged_filtered_molecules.csv") 
output_keep_strict = os.path.join(current_dir, "sos1_strict_"+basename+"_flaged_filtered_molecules.csv") 

df_all, df_hard, df_strict, df_incomplete = run_filter_and_save(
    input_file,
    vina_score=-8.0,
    cns_target=False,                       # CNS 项目
    stage="screen",
    # output_all=output_file,
    output_keep_hard=output_keep_hard,
    output_keep_strict=output_keep_strict,
    output_incomplete=f"sos1_incomplete_{basename}_records.csv",
)
print(sum(df_hard['BBB_pref'] == 0) / len(df_hard) * 100)
print(sum(df_hard['Caco2_pref'] == 0) / len(df_hard) * 100)

🔍 检查的关键列（共 9 个）: ['aff', 'hERG', 'DILI', 'AMES', 'CYP3A4_Veith', 'CYP2D6_Veith', 'HIA_Hou', 'Caco2_Wang', 'BBB_Martins']
📊 总行数: 10000 | 完整行: 10000 | 缺失关键列: 0
💾 已保存 0 条 incomplete 记录到: sos1_incomplete_generate_oral_pg_hpk1_admet_39_dyn_oral_policy_200_20251107_115825_0_hHPK1_vina_admet_records.csv
✅ 已保存 keep_hard (288 行) 到: /raid/home/xukai/FRATTVAE_Analysis/sos1_hard_generate_oral_pg_hpk1_admet_39_dyn_oral_policy_200_20251107_115825_0_hHPK1_vina_admet_flaged_filtered_molecules.csv
✅ 已保存 keep_strict (36 行) 到: /raid/home/xukai/FRATTVAE_Analysis/sos1_strict_generate_oral_pg_hpk1_admet_39_dyn_oral_policy_200_20251107_115825_0_hHPK1_vina_admet_flaged_filtered_molecules.csv
5.208333333333334
64.93055555555556


In [17]:
input_file = "/raid/home/xukai/FRATTVAE/results/chembl_36_20251023_r1r10w1100w900_standardized_f_dyn_oral_struct_1031/generate/generate_oral_pg_hpk1_admet_39_dyn_oral_policy_200_20251107_115825_0_hHPK1_vina_admet.csv"
current_dir = os.getcwd()
basename = os.path.splitext(os.path.basename(input_file))[0]
# output_file = os.path.join(current_dir, basename+"_flaged_filtered_molecules_lrrk2.csv") 
output_keep_hard = os.path.join(current_dir, "sos1_hard_"+basename+"_flaged_filtered_molecules.csv") 
output_keep_strict = os.path.join(current_dir, "sos1_strict_"+basename+"_flaged_filtered_molecules.csv") 

df_all, df_hard, df_strict, df_incomplete = run_filter_and_save(
    input_file,
    vina_score=-8.0,
    cns_target=False,                       # CNS 项目
    stage="candidate",
    # output_all=output_file,
    output_keep_hard=output_keep_hard,
    output_keep_strict=output_keep_strict,
    output_incomplete=f"sos1_incomplete_{basename}_records.csv",
)
print(sum(df_hard['BBB_pref'] == 0) / len(df_hard) * 100)
print(sum(df_hard['Caco2_pref'] == 0) / len(df_hard) * 100)

🔍 检查的关键列（共 9 个）: ['aff', 'hERG', 'DILI', 'AMES', 'CYP3A4_Veith', 'CYP2D6_Veith', 'HIA_Hou', 'Caco2_Wang', 'BBB_Martins']
📊 总行数: 10000 | 完整行: 10000 | 缺失关键列: 0
💾 已保存 0 条 incomplete 记录到: sos1_incomplete_generate_oral_pg_hpk1_admet_39_dyn_oral_policy_200_20251107_115825_0_hHPK1_vina_admet_records.csv
✅ 已保存 keep_hard (279 行) 到: /raid/home/xukai/FRATTVAE_Analysis/sos1_hard_generate_oral_pg_hpk1_admet_39_dyn_oral_policy_200_20251107_115825_0_hHPK1_vina_admet_flaged_filtered_molecules.csv
✅ 已保存 keep_strict (17 行) 到: /raid/home/xukai/FRATTVAE_Analysis/sos1_strict_generate_oral_pg_hpk1_admet_39_dyn_oral_policy_200_20251107_115825_0_hHPK1_vina_admet_flaged_filtered_molecules.csv
5.017921146953405
64.51612903225806


In [15]:
import os
input_file = "/raid/home/xukai/FRATTVAE/results/CNS_SMILES_standardized_struct_1020/generate/generate_best_1_hHPK1_properties_updated_admet.csv"
current_dir = os.getcwd()
basename = os.path.splitext(os.path.basename(input_file))[0]
# output_file = os.path.join(current_dir, basename+"_flaged_filtered_molecules_lrrk2.csv") 
output_keep_hard = os.path.join(current_dir, "sos1_hard_"+basename+"_flaged_filtered_molecules.csv") 
output_keep_strict = os.path.join(current_dir, "sos1_strict_"+basename+"_flaged_filtered_molecules.csv") 

df_all, df_hard, df_strict, df_incomplete = run_filter_and_save(
    input_file,
    vina_score=-8.0,
    cns_target=False,                       # CNS 项目
    stage="screen",
    # output_all=output_file,
    output_keep_hard=output_keep_hard,
    output_keep_strict=output_keep_strict,
    output_incomplete=f"sos1_incomplete_{basename}_records.csv",
)
df_all, df_hard, df_strict, df_incomplete = run_filter_and_save(
    input_file,
    vina_score=-8.0,
    cns_target=False,                       # CNS 项目
    stage="candidate",
    # output_all=output_file,
    output_keep_hard=output_keep_hard,
    output_keep_strict=output_keep_strict,
    output_incomplete=f"sos1_incomplete_{basename}_records.csv",
)
print(sum(df_hard['BBB_pref'] == 0) / len(df_hard) * 100)
print(sum(df_hard['Caco2_pref'] == 0) / len(df_hard) * 100)     


🔍 检查的关键列（共 9 个）: ['aff', 'hERG', 'DILI', 'AMES', 'CYP3A4_Veith', 'CYP2D6_Veith', 'HIA_Hou', 'Caco2_Wang', 'BBB_Martins']
📊 总行数: 10018 | 完整行: 10018 | 缺失关键列: 0
💾 已保存 0 条 incomplete 记录到: sos1_incomplete_generate_best_1_hHPK1_properties_updated_admet_records.csv
✅ 已保存 keep_hard (48 行) 到: /raid/home/xukai/FRATTVAE_Analysis/sos1_hard_generate_best_1_hHPK1_properties_updated_admet_flaged_filtered_molecules.csv
✅ 已保存 keep_strict (0 行) 到: /raid/home/xukai/FRATTVAE_Analysis/sos1_strict_generate_best_1_hHPK1_properties_updated_admet_flaged_filtered_molecules.csv
🔍 检查的关键列（共 9 个）: ['aff', 'hERG', 'DILI', 'AMES', 'CYP3A4_Veith', 'CYP2D6_Veith', 'HIA_Hou', 'Caco2_Wang', 'BBB_Martins']
📊 总行数: 10018 | 完整行: 10018 | 缺失关键列: 0
💾 已保存 0 条 incomplete 记录到: sos1_incomplete_generate_best_1_hHPK1_properties_updated_admet_records.csv
✅ 已保存 keep_hard (48 行) 到: /raid/home/xukai/FRATTVAE_Analysis/sos1_hard_generate_best_1_hHPK1_properties_updated_admet_flaged_filtered_molecules.csv
✅ 已保存 keep_strict (0 行) 到: /raid/ho